In [0]:
WITH ids AS (
  SELECT member_id AS CDE_ID FROM data_exploration_hub.marketing_hub.t1_matched_pairs
  UNION
  SELECT lookalike_id FROM data_exploration_hub.marketing_hub.t1_matched_pairs
),
theatres AS (
  SELECT DISTINCT LocationID FROM cpx_dataanalytics_gold.customer360.cpx_d_location
  WHERE LC_Location_Type_Description = 'Cineplex Theatre'
),
tx AS (
  SELECT CAST(t.CDE_ID AS INT) AS CDE_ID, t.VisitDateId, t.LocationId, t.Net_Revenue,
         t.TicketFLAG, t.ConcessionFLAG, t.Quantity, s.Auditorium_Experience_ID
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN theatres l ON t.LocationId = l.LocationID
  JOIN ids i      ON CAST(t.CDE_ID AS INT) = i.CDE_ID
  LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s ON t.SessionSkey = s.SessionSkey
  WHERE t.VisitDateId >= 20260201 AND t.VisitDateId < 20260801          -- Feb 1 - Jul 31, 2026
    AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No'                     -- NEW: BR-011
),
v AS (                  -- visit = CDE + date + location
  SELECT CDE_ID, VisitDateId, LocationId,
         sum(Net_Revenue)                                                          AS rev,
         sum(CASE WHEN TicketFLAG = 'Yes' THEN Net_Revenue ELSE 0 END)             AS ticket_rev,
         sum(CASE WHEN ConcessionFLAG = 'Yes' THEN Net_Revenue ELSE 0 END)         AS conc_rev,
         sum(CASE WHEN TicketFLAG = 'Yes' THEN Quantity ELSE 0 END)                AS net_tickets,
         sum(CASE WHEN TicketFLAG = 'Yes' AND Auditorium_Experience_ID IN (2, 3, 4, 7, 8)
                  THEN Quantity ELSE 0 END)                                        AS prem_tickets,   -- AVX, IMAX, VIP, 4DX, ScreenX
         sum(CASE WHEN TicketFLAG = 'Yes' AND Auditorium_Experience_ID = 1
                  THEN Quantity ELSE 0 END)                                        AS reg_tickets,    -- NEW: KPI-19 regular
         max(CASE WHEN ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END)                   AS has_conc
  FROM tx
  GROUP BY CDE_ID, VisitDateId, LocationId
),
typed AS (
  SELECT *,
         (net_tickets > 0 AND rev > 0)                                                    AS is_movie,
         (net_tickets > 0 AND rev > 0) OR (net_tickets <= 0 AND has_conc = 1 AND rev > 0) AS is_visit
  FROM v
),
p AS (
  SELECT CDE_ID,
         count_if(is_visit)                                   AS visits,
         count_if(is_movie)                                   AS movie_visits,
         sum(rev)                                             AS total_rev,
         sum(ticket_rev)                                      AS ticket_rev,
         sum(conc_rev)                                        AS conc_rev,
         count_if(is_visit AND has_conc = 1)                  AS visits_w_conc,
         sum(CASE WHEN is_movie THEN net_tickets ELSE 0 END)  AS tickets,
         sum(CASE WHEN is_movie THEN prem_tickets ELSE 0 END) AS prem_tickets,
         sum(CASE WHEN is_movie THEN reg_tickets ELSE 0 END)  AS reg_tickets
  FROM typed
  GROUP BY CDE_ID
),
k AS (                  -- every matched person; no visits = zeros
  SELECT i.CDE_ID,
         CAST(coalesce(p.visits, 0)        AS DOUBLE) AS visits,
         CAST(coalesce(p.movie_visits, 0)  AS DOUBLE) AS movie_visits,
         CAST(coalesce(p.total_rev, 0)     AS DOUBLE) AS total_rev,
         CAST(coalesce(p.ticket_rev, 0)    AS DOUBLE) AS ticket_rev,
         CAST(coalesce(p.conc_rev, 0)      AS DOUBLE) AS conc_rev,
         CAST(coalesce(p.visits_w_conc, 0) AS DOUBLE) AS visits_w_conc,
         CAST(coalesce(p.tickets, 0)       AS DOUBLE) AS tickets,
         CAST(coalesce(p.prem_tickets, 0)  AS DOUBLE) AS prem_tickets,
         CAST(coalesce(p.reg_tickets, 0)   AS DOUBLE) AS reg_tickets,
         CAST(CASE WHEN coalesce(p.prem_tickets, 0) > 0 THEN 1 ELSE 0 END AS DOUBLE) AS used_premium,
         CAST(CASE WHEN coalesce(p.visits, 0) > 0 THEN 1 ELSE 0 END AS DOUBLE)       AS has_visit     -- NEW: KPI-12 base
  FROM ids i
  LEFT JOIN p ON p.CDE_ID = i.CDE_ID
),
lk AS (                 -- each member's 5 lookalikes averaged (reuse = weight)
  SELECT pr.member_id,
         avg(k.visits) AS visits, avg(k.movie_visits) AS movie_visits, avg(k.total_rev) AS total_rev,
         avg(k.ticket_rev) AS ticket_rev, avg(k.conc_rev) AS conc_rev, avg(k.visits_w_conc) AS visits_w_conc,
         avg(k.tickets) AS tickets, avg(k.prem_tickets) AS prem_tickets, avg(k.reg_tickets) AS reg_tickets,
         avg(k.used_premium) AS used_premium, avg(k.has_visit) AS has_visit
  FROM data_exploration_hub.marketing_hub.t1_matched_pairs pr
  JOIN k ON k.CDE_ID = pr.lookalike_id
  GROUP BY pr.member_id
),
mm AS (
  SELECT l.member_id,
         CASE WHEN f.cc_status = 'Active' THEN 1 ELSE 0 END AS active_aug,
         m.visits AS m_visits, m.movie_visits AS m_movie_visits, m.total_rev AS m_total_rev,
         m.ticket_rev AS m_ticket_rev, m.conc_rev AS m_conc_rev, m.visits_w_conc AS m_visits_w_conc,
         m.tickets AS m_tickets, m.prem_tickets AS m_prem_tickets, m.reg_tickets AS m_reg_tickets,
         m.used_premium AS m_used_premium, m.has_visit AS m_has_visit,
         l.visits AS l_visits, l.movie_visits AS l_movie_visits, l.total_rev AS l_total_rev,
         l.ticket_rev AS l_ticket_rev, l.conc_rev AS l_conc_rev, l.visits_w_conc AS l_visits_w_conc,
         l.tickets AS l_tickets, l.prem_tickets AS l_prem_tickets, l.reg_tickets AS l_reg_tickets,
         l.used_premium AS l_used_premium, l.has_visit AS l_has_visit
  FROM lk l
  JOIN k m ON m.CDE_ID = l.member_id
  LEFT JOIN data_exploration_hub.marketing_hub.propensity_features f
    ON f.CDE_ID = l.member_id AND f.as_of_date = DATE'2026-08-01'
),
views AS (
  SELECT '1 Active on Aug 1 (headline)' AS view, * FROM mm WHERE active_aug = 1
  UNION ALL
  SELECT '2 All matched members', * FROM mm
),
agg AS (
  SELECT view, count(*) AS n,
         sum(m_visits) AS m_visits, sum(m_movie_visits) AS m_movie_visits, sum(m_total_rev) AS m_total_rev,
         sum(m_ticket_rev) AS m_ticket_rev, sum(m_conc_rev) AS m_conc_rev, sum(m_visits_w_conc) AS m_visits_w_conc,
         sum(m_tickets) AS m_tickets, sum(m_prem_tickets) AS m_prem_tickets, sum(m_reg_tickets) AS m_reg_tickets,
         sum(m_used_premium) AS m_used_premium, sum(m_has_visit) AS m_has_visit,
         sum(l_visits) AS l_visits, sum(l_movie_visits) AS l_movie_visits, sum(l_total_rev) AS l_total_rev,
         sum(l_ticket_rev) AS l_ticket_rev, sum(l_conc_rev) AS l_conc_rev, sum(l_visits_w_conc) AS l_visits_w_conc,
         sum(l_tickets) AS l_tickets, sum(l_prem_tickets) AS l_prem_tickets, sum(l_reg_tickets) AS l_reg_tickets,
         sum(l_used_premium) AS l_used_premium, sum(l_has_visit) AS l_has_visit,
         stddev(m_has_visit - l_has_visit) / sqrt(count(*)) AS se_reach
  FROM views
  GROUP BY view
),
kpi AS (
  SELECT view, n, stack(12,
    '00 Reach: % with >= 1 visit',            100 * m_has_visit / n,                          100 * l_has_visit / n,                          100 * se_reach,
    '01 Avg frequency (visits / active person)', m_visits / m_has_visit,                      l_visits / l_has_visit,                         CAST(NULL AS DOUBLE),
    '02 Total spend per visit ($)',           m_total_rev / m_visits,                         l_total_rev / l_visits,                         CAST(NULL AS DOUBLE),
    '03 Ticket spend per movie visit ($)',    m_ticket_rev / m_movie_visits,                  l_ticket_rev / l_movie_visits,                  CAST(NULL AS DOUBLE),
    '04 Concession spend per visit ($)',      m_conc_rev / m_visits,                          l_conc_rev / l_visits,                          CAST(NULL AS DOUBLE),
    '05 Total spend per active person, 6M ($)',      m_total_rev / m_has_visit,               l_total_rev / l_has_visit,                      CAST(NULL AS DOUBLE),
    '06 Ticket spend per active person, 6M ($)',     m_ticket_rev / m_has_visit,              l_ticket_rev / l_has_visit,                     CAST(NULL AS DOUBLE),
    '07 Concession spend per active person, 6M ($)', m_conc_rev / m_has_visit,                l_conc_rev / l_has_visit,                       CAST(NULL AS DOUBLE),
    '08 Concession incidence (% of visits)',  100 * m_visits_w_conc / m_visits,               100 * l_visits_w_conc / l_visits,               CAST(NULL AS DOUBLE),
    '09 Tickets per movie visit',             m_tickets / m_movie_visits,                     l_tickets / l_movie_visits,                     CAST(NULL AS DOUBLE),
    '10 Premium share (% of premium + regular tickets)', 100 * m_prem_tickets / (m_prem_tickets + m_reg_tickets),
                                                         100 * l_prem_tickets / (l_prem_tickets + l_reg_tickets), CAST(NULL AS DOUBLE),
    '11 Active people using premium (%)',     100 * m_used_premium / m_has_visit,             100 * l_used_premium / l_has_visit,             CAST(NULL AS DOUBLE)
  ) AS (kpi, members, lookalikes, se)
  FROM agg
)
SELECT view, n AS members_n, kpi,
       round(members, 2)                                   AS members,
       round(lookalikes, 2)                                AS lookalikes,
       round(members - lookalikes, 2)                      AS lift,
       round(100 * (members - lookalikes) / lookalikes, 1) AS lift_pct,
       round(members - lookalikes - 1.96 * se, 2)          AS ci95_low,
       round(members - lookalikes + 1.96 * se, 2)          AS ci95_high
FROM kpi
ORDER BY view, kpi;